# Reproducible analysis of two TikTok datasets

This notebook checks the downloaded **U.S. 2024 Presidential Election TikTok IDs** and **News on TikTok (GESIS)** datasets. It examines file structure, metadata, reconstructability, release counts, paper-count agreement, missing values, engagement statistics, and similarities and differences.

The notebook does not download videos or contact TikTok by default. The U.S. overlap checks process millions of IDs and can take several minutes.

## Sources and important interpretation notes

- U.S. paper: [Tracking the 2024 U.S. Presidential Election Chatter on TikTok](https://arxiv.org/abs/2412.15583)
- U.S. dataset repository: [US2024PresElectionTikToks](https://github.com/gabbypinto/US2024PresElectionTikToks)
- GESIS paper: [News on TikTok](https://ojs.aaai.org/index.php/ICWSM/article/view/35953)
- GESIS dataset record: [doi:10.7802/2819](https://doi.org/10.7802/2819)

A video ID is not raw video content. IDs may support later rehydration, but videos can become private or deleted. The public U.S. CSV files contain IDs only. GESIS contains IDs, URLs, metadata, and human annotations, but not video/audio files, captions, transcripts, OCR, or comment text.

In [1]:
from pathlib import Path
import ast
import csv
import re
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent

US_DIR = ROOT / 'downloaded_datasets' / 'us_election'
GESIS_DIR = ROOT / 'downloaded_datasets' / 'gesis_news'
US_LATEST = US_DIR / 'TikTok_IDs_2025_Feb.csv'
US_MEMO1 = US_DIR / 'TikTok_IDs_Memo1.csv'
US_V2 = US_DIR / 'TikTok_IDs_v2.csv'
GESIS_CSV = GESIS_DIR / 'TikTok-news_data_v2-0-0.csv'

required = [US_LATEST, US_MEMO1, US_V2, GESIS_CSV]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing dataset files:\n' + '\n'.join(missing))

print('Project root:', ROOT)
print('All four CSV files were found.')

Project root: /Users/wang/Documents/tiktok
All four CSV files were found.


## 1. File inventory and schemas

In [2]:
def mib(path):
    return path.stat().st_size / (1024 ** 2)

inventory = pd.DataFrame([
    {'dataset': 'US election', 'release': 'Memo1', 'file': US_MEMO1.name, 'size_MiB': mib(US_MEMO1)},
    {'dataset': 'US election', 'release': 'v2', 'file': US_V2.name, 'size_MiB': mib(US_V2)},
    {'dataset': 'US election', 'release': '2025 Feb', 'file': US_LATEST.name, 'size_MiB': mib(US_LATEST)},
    {'dataset': 'News on TikTok', 'release': 'v2.0.0', 'file': GESIS_CSV.name, 'size_MiB': mib(GESIS_CSV)},
])
inventory.loc['Total', 'size_MiB'] = inventory['size_MiB'].sum()
display(inventory.round({'size_MiB': 2}))

,dataset,release,file,size_MiB
0,US election,Memo1,TikTok_IDs_Memo1.csv,34.32
1,US election,v2,TikTok_IDs_v2.csv,89.49
2,US election,2025 Feb,TikTok_IDs_2025_Feb.csv,77.63
3,News on TikTok,v2.0.0,TikTok-news_data_v2-0-0.csv,2.91
Total,NaN,NaN,NaN,204.35


In [3]:
for path in required:
    preview = pd.read_csv(path, nrows=3, dtype=str)
    print(f'\n{path.name}: {preview.shape[1]} columns')
    display(preview)


TikTok_IDs_2025_Feb.csv: 1 columns


,id
0,7409828458894789931
1,7409828193252756779
2,7409827326789356846



TikTok_IDs_Memo1.csv: 1 columns


,id
0,7298132049062382891
1,7298132047581924654
2,7298131955592465710



TikTok_IDs_v2.csv: 2 columns


,Unnamed: 0,id
0,0,7.297389872858566e+18
1,1,7.297389853237317e+18
2,2,7.297389825458441e+18



TikTok-news_data_v2-0-0.csv: 37 columns


,V1_coder,V2_video_link,V3_news_outlet,V4_country_of_origin,V5_news_classification,V6_visual_set-up,V7a_images,V7b_text,V7c_screenshots_and_recordings,V7d_graphs_maps_infoanimations,V7e_news_brand_elements,V7f_outlet-related_endcard,V8a_journalists_moderators_voice,V8b_other_peoples_voice,V8c_voice_other_effect,V8d_music,V8e_ambient,V9a_mentions,V9b_call_to_interact_on_tiktok,V9c_call_to_interact_outside_of_tiktok,V10_news_format_type,V11_general_topic,V12_geographic_scope,V13_follow_up,V14_positive_news,V15_negative_news,V16_conflict,V17_scope,V18_temporal_scope,M1_video_id,M2_music_id,M3_create_time,M4_like_count,M5_view_count,M6_comment_count,M7_share_count,M8_hashtags
0,1,https://www.tiktok.com/share/video/72033756326...,tagesanzeiger,CH,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,Erklärungen zur Steuererklärung,1.0,0.0,0.0,0.0,0.0,1.0,2.0,7203375632661695749,7203375660541807365,2023-02-23 15:36:28,38,680.0,1,3,"['tagi', 'steuern', 'tagesanzeiger']"
1,0,https://www.tiktok.com/share/video/72029786320...,tagesanzeiger,CH,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,Vanilla Girl Trend,0.0,0.0,0.0,0.0,1.0,1.0,2.0,7202978632086670598,7202978669403998982,2023-02-22 13:55:55,32,653.0,0,2,"['people', 'climatechange', 'vanillagirl', 'zü..."
2,0,https://www.tiktok.com/share/video/72018384221...,tagesanzeiger,CH,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,Hacker-Team Destabilisation von Deomkratien ge...,2.0,0.0,0.0,1.0,0.0,1.0,0.0,7201838422108032262,7086563693013518338,2023-02-19 12:11:22,45,521.0,1,0,"['news', 'tagi', 'teamjorge', 'tagesanzeiger']"


## 2. U.S. election dataset

The public release contains only TikTok video IDs. The paper describes a richer internal research corpus, but that metadata is not present in these CSV files. The cells below preserve IDs as strings and never parse valid IDs as floating-point numbers.

In [4]:
EXACT_ID = re.compile(r'^\d{19}$')
SCIENTIFIC_ID = re.compile(r'^\d(?:\.\d+)?e\+18$', re.IGNORECASE)

def profile_id_csv(path, id_column='id', chunksize=500_000):
    rows = exact_19_digit = scientific = missing = 0
    unique_values = set()
    for chunk in pd.read_csv(path, dtype=str, usecols=[id_column], chunksize=chunksize):
        values = chunk[id_column]
        rows += len(values)
        missing += int(values.isna().sum())
        clean = values.dropna().str.strip()
        exact_19_digit += int(clean.str.fullmatch(EXACT_ID).sum())
        scientific += int(clean.str.fullmatch(SCIENTIFIC_ID).sum())
        unique_values.update(clean.tolist())
    return {
        'file': path.name,
        'rows': rows,
        'unique_text_values': len(unique_values),
        'duplicate_rows': rows - missing - len(unique_values),
        'missing_ids': missing,
        'exact_19_digit_ids': exact_19_digit,
        'scientific_notation_ids': scientific,
    }

us_profiles = pd.DataFrame([
    profile_id_csv(US_MEMO1),
    profile_id_csv(US_V2),
    profile_id_csv(US_LATEST),
])
display(us_profiles)

,file,rows,unique_text_values,duplicate_rows,missing_ids,exact_19_digit_ids,scientific_notation_ids
0,TikTok_IDs_Memo1.csv,1799333,1799333,0,0,1799333,0
1,TikTok_IDs_v2.csv,3176949,3176949,0,0,0,3176948
2,TikTok_IDs_2025_Feb.csv,4069908,4069908,0,0,4069908,0


### Compare release counts with the paper and repository

The paper snapshot reports 3,144,836 videos. The repository later reports 3,176,949 IDs for the November 2024 release and 4,069,908 IDs for the February 2025 release. Different dates and coverage windows explain why these values should not be identical.

In [5]:
US_PAPER_VIDEO_COUNT = 3_144_836
US_PAPER_USER_COUNT = 693_826
US_PAPER_TRANSCRIPT_COUNT = 485_343

release_counts = pd.DataFrame([
    {'snapshot': 'Paper', 'date_or_window': 'posts through 16 Oct 2024', 'count': US_PAPER_VIDEO_COUNT},
    {'snapshot': 'Repository v2', 'date_or_window': 'released 1 Nov 2024', 'count': int(us_profiles.loc[us_profiles.file == US_V2.name, 'rows'].iloc[0])},
    {'snapshot': 'Repository Feb 2025', 'date_or_window': 'posts through 20 Jan 2025', 'count': int(us_profiles.loc[us_profiles.file == US_LATEST.name, 'rows'].iloc[0])},
])
release_counts['difference_from_paper'] = release_counts['count'] - US_PAPER_VIDEO_COUNT
release_counts['percent_difference_from_paper'] = release_counts['difference_from_paper'] / US_PAPER_VIDEO_COUNT * 100
display(release_counts.round({'percent_difference_from_paper': 2}))

,snapshot,date_or_window,count,difference_from_paper,percent_difference_from_paper
0,Paper,posts through 16 Oct 2024,3144836,0,0.00
1,Repository v2,released 1 Nov 2024,3176949,32113,1.02
2,Repository Feb 2025,posts through 20 Jan 2025,4069908,925072,29.42


### Overlap between Memo1 and the February 2025 release

This stores the smaller Memo1 ID set in memory and streams through the latest file. The files overlap, so they must not be concatenated and counted as independent videos.

In [6]:
def load_id_set(path, id_column='id', chunksize=500_000):
    result = set()
    for chunk in pd.read_csv(path, dtype=str, usecols=[id_column], chunksize=chunksize):
        result.update(chunk[id_column].dropna().str.strip())
    return result

memo_ids = load_id_set(US_MEMO1)
memo_in_latest = 0
latest_rows = 0
for chunk in pd.read_csv(US_LATEST, dtype=str, usecols=['id'], chunksize=500_000):
    ids = chunk['id'].dropna().str.strip()
    latest_rows += len(ids)
    memo_in_latest += int(ids.isin(memo_ids).sum())

overlap_summary = pd.DataFrame([{
    'Memo1 unique IDs': len(memo_ids),
    'Memo1 IDs also in Feb release': memo_in_latest,
    'Memo1 IDs absent from Feb release': len(memo_ids) - memo_in_latest,
    'Memo1 overlap percent': memo_in_latest / len(memo_ids) * 100,
    'Feb release rows': latest_rows,
}])
display(overlap_summary.round({'Memo1 overlap percent': 2}))
del memo_ids

,Memo1 unique IDs,Memo1 IDs also in Feb release,Memo1 IDs absent from Feb release,Memo1 overlap percent,Feb release rows
0,1799333,1294289,505044,71.93,4069908


### ID reconstructability and the v2 precision problem

TikTok IDs are 19-digit identifiers and must be stored as text or integer values. In the downloaded `v2` CSV, almost every ID is written in scientific notation. A value such as `7.297389872858566e+18` contains fewer significant digits than the original 19-digit ID, so its exact trailing digits cannot be recovered reliably.

In [7]:
v2_sample = pd.read_csv(US_V2, dtype=str, nrows=10)
latest_sample = pd.read_csv(US_LATEST, dtype=str, nrows=10)
display(pd.DataFrame({
    'v2 stored text': v2_sample['id'],
    'unsafe float-to-integer conversion': v2_sample['id'].map(lambda x: str(int(float(x))) if pd.notna(x) else None),
    'latest exact text': latest_sample['id'],
}))

print('Conclusion: Memo1 and the February file contain exact 19-digit IDs.')
print('The v2 file is not safe for exact video reconstruction because precision was lost.')
print('The public U.S. files do not include creator usernames, so the repository download script cannot build its expected username/video URL directly from these CSVs.')

,v2 stored text,unsafe float-to-integer conversion,latest exact text
0,7.297389872858566e+18,7297389872858565632,7409828458894789931
1,7.297389853237317e+18,7297389853237316608,7409828193252756779
2,7.297389825458441e+18,7297389825458441216,7409827326789356846
3,7.297389728679202e+18,7297389728679201792,7409827227283557675
4,7.297389700514319e+18,7297389700514319360,7409827072966692139
5,7.297389651285888e+18,7297389651285888000,7409826951856311598
6,7.297389634219265e+18,7297389634219265024,7409825915842071854
7,7.297389628678524e+18,7297389628678523904,7409825794702298414
8,7.297389618176003e+18,7297389618176003072,7409825669691067691
9,7.297389609275723e+18,7297389609275722752,7409825176818339115


Conclusion: Memo1 and the February file contain exact 19-digit IDs.
The v2 file is not safe for exact video reconstruction because precision was lost.
The public U.S. files do not include creator usernames, so the repository download script cannot build its expected username/video URL directly from these CSVs.


## 3. News on TikTok (GESIS) dataset

In [8]:
gesis = pd.read_csv(
    GESIS_CSV,
    dtype={'M1_video_id': 'string', 'M2_music_id': 'string'},
)
gesis['M3_create_time'] = pd.to_datetime(gesis['M3_create_time'], errors='coerce')

print('Shape:', gesis.shape)
print('Unique outlets:', gesis['V3_news_outlet'].nunique())
print('Unique video IDs:', gesis['M1_video_id'].nunique())
print('Unique video URLs:', gesis['V2_video_link'].nunique())
print('Publication window:', gesis['M3_create_time'].min(), 'to', gesis['M3_create_time'].max())
display(gesis.head(3))

Shape: (8623, 37)
Unique outlets: 18
Unique video IDs: 8623
Unique video URLs: 8623
Publication window: 2023-01-01 10:24:12 to 2023-12-31 23:00:00


,V1_coder,V2_video_link,V3_news_outlet,V4_country_of_origin,V5_news_classification,V6_visual_set-up,V7a_images,V7b_text,V7c_screenshots_and_recordings,V7d_graphs_maps_infoanimations,V7e_news_brand_elements,V7f_outlet-related_endcard,V8a_journalists_moderators_voice,V8b_other_peoples_voice,V8c_voice_other_effect,V8d_music,V8e_ambient,V9a_mentions,V9b_call_to_interact_on_tiktok,V9c_call_to_interact_outside_of_tiktok,V10_news_format_type,V11_general_topic,V12_geographic_scope,V13_follow_up,V14_positive_news,V15_negative_news,V16_conflict,V17_scope,V18_temporal_scope,M1_video_id,M2_music_id,M3_create_time,M4_like_count,M5_view_count,M6_comment_count,M7_share_count,M8_hashtags
0,1,https://www.tiktok.com/share/video/72033756326...,tagesanzeiger,CH,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,2.00,Erklärungen zur Steuererklärung,1.00,0.00,0.00,0.00,0.00,1.00,2.00,7203375632661695749,7203375660541807365,2023-02-23 15:36:28,38,680.00,1,3,"['tagi', 'steuern', 'tagesanzeiger']"
1,0,https://www.tiktok.com/share/video/72029786320...,tagesanzeiger,CH,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,3.00,Vanilla Girl Trend,0.00,0.00,0.00,0.00,1.00,1.00,2.00,7202978632086670598,7202978669403998982,2023-02-22 13:55:55,32,653.00,0,2,"['people', 'climatechange', 'vanillagirl', 'zü..."
2,0,https://www.tiktok.com/share/video/72018384221...,tagesanzeiger,CH,1.00,1.00,1.00,1.00,1.00,0.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,2.00,Hacker-Team Destabilisation von Deomkratien ge...,2.00,0.00,0.00,1.00,0.00,1.00,0.00,7201838422108032262,7086563693013518338,2023-02-19 12:11:22,45,521.00,1,0,"['news', 'tagi', 'teamjorge', 'tagesanzeiger']"


### Metadata structure and missing values

In [9]:
field_groups = pd.DataFrame([
    {'group': 'Human annotations', 'columns': 'V1-V18', 'included': 'Yes', 'examples': 'news class, visual setup, audio setup, format, topic, scope'},
    {'group': 'Identifiers and links', 'columns': 'V2, M1-M2', 'included': 'Yes', 'examples': 'video URL, video ID, music ID'},
    {'group': 'Post metadata', 'columns': 'V3-V4, M3, M8', 'included': 'Yes', 'examples': 'outlet, country, creation time, hashtags'},
    {'group': 'Engagement counters', 'columns': 'M4-M7', 'included': 'Yes', 'examples': 'likes, views, comments, shares'},
    {'group': 'Raw media', 'columns': '-', 'included': 'No', 'examples': 'no video or audio files'},
    {'group': 'Derived media text', 'columns': '-', 'included': 'No', 'examples': 'no captions, transcripts, ASR, or OCR'},
    {'group': 'Comments', 'columns': 'M6 only', 'included': 'Count only', 'examples': 'no comment text or comment rows'},
])
display(field_groups)

missingness = pd.DataFrame({
    'missing_count': gesis.isna().sum(),
    'missing_percent': gesis.isna().mean() * 100,
}).sort_values(['missing_count', 'missing_percent'], ascending=False)
display(missingness[missingness['missing_count'] > 0].round(2))

,group,columns,included,examples
0,Human annotations,V1-V18,Yes,"news class, visual setup, audio setup, format,..."
1,Identifiers and links,"V2, M1-M2",Yes,"video URL, video ID, music ID"
2,Post metadata,"V3-V4, M3, M8",Yes,"outlet, country, creation time, hashtags"
3,Engagement counters,M4-M7,Yes,"likes, views, comments, shares"
4,Raw media,-,No,no video or audio files
5,Derived media text,-,No,"no captions, transcripts, ASR, or OCR"
6,Comments,M6 only,Count only,no comment text or comment rows


,missing_count,missing_percent
V7d_graphs_maps_infoanimations,646,7.49
V6_visual_set-up,645,7.48
V7a_images,645,7.48
V7b_text,645,7.48
V7c_screenshots_and_recordings,645,7.48
V7e_news_brand_elements,645,7.48
V7f_outlet-related_endcard,645,7.48
V8a_journalists_moderators_voice,645,7.48
V8b_other_peoples_voice,645,7.48
V8c_voice_other_effect,645,7.48


### Check whether URLs and IDs support reconstruction

GESIS provides both the original TikTok URL and the exact video ID. This is stronger than an ID-only release, although deleted or private videos still cannot be recovered.

In [10]:
url_id = gesis['V2_video_link'].astype('string').str.extract(r'(\d{19})(?:\D*)$', expand=False)
id_checks = pd.DataFrame([{
    'rows': len(gesis),
    'unique_IDs': gesis['M1_video_id'].nunique(),
    'duplicate_ID_rows': int(gesis['M1_video_id'].duplicated().sum()),
    'unique_URLs': gesis['V2_video_link'].nunique(),
    'URL_ID_matches': int((url_id == gesis['M1_video_id']).sum()),
    'URL_ID_mismatches_or_missing': int((url_id != gesis['M1_video_id']).fillna(True).sum()),
}])
display(id_checks)

,rows,unique_IDs,duplicate_ID_rows,unique_URLs,URL_ID_matches,URL_ID_mismatches_or_missing
0,8623,8623,0,8623,8623,0


### Compare downloaded counts with the paper

In [11]:
news_class_counts = gesis['V5_news_classification'].value_counts(dropna=False).sort_index()
count_checks = pd.DataFrame([
    {'measure': 'All videos', 'paper': 8_623, 'downloaded': len(gesis)},
    {'measure': 'Outlet accounts', 'paper': 18, 'downloaded': gesis['V3_news_outlet'].nunique()},
    {'measure': 'Classified as news (V5=1)', 'paper': 7_978, 'downloaded': int((gesis['V5_news_classification'] == 1).sum())},
])
count_checks['difference'] = count_checks['downloaded'] - count_checks['paper']
count_checks['identical'] = count_checks['difference'].eq(0)
display(count_checks)

classification_labels = {
    1.0: 'news', 2.0: 'self-promotion', 3.0: 'branded content',
    4.0: 'interactive content', 77.0: 'unavailable', -99.0: 'unclear'
}
classification = news_class_counts.rename(index=classification_labels).rename('count').to_frame()
classification['percent'] = classification['count'] / len(gesis) * 100
display(classification.round(2))

,measure,paper,downloaded,difference,identical
0,All videos,8623,8623,0,True
1,Outlet accounts,18,18,0,True
2,Classified as news (V5=1),7978,7978,0,True


,count,percent
V5_news_classification,,
unclear,111,1.29
news,7978,92.52
self-promotion,172,1.99
branded content,187,2.17
interactive content,106,1.23
unavailable,69,0.80


### Reproduce the paper's engagement table

Pandas uses the sample standard deviation (`ddof=1`), matching the reported table. Notice that the view-count mean uses 8,559 non-missing values because 64 rows have no view count.

In [12]:
engagement_columns = {
    'Views': 'M5_view_count',
    'Likes': 'M4_like_count',
    'Comments': 'M6_comment_count',
    'Shares': 'M7_share_count',
}
paper_engagement = pd.DataFrame({
    'mean': {'Views': 48_878.32, 'Likes': 4_523.64, 'Comments': 200.67, 'Shares': 303.34},
    'std': {'Views': 191_810.48, 'Likes': 17_199.79, 'Comments': 1_655.49, 'Shares': 1_854.64},
    'median': {'Views': 1_645, 'Likes': 707, 'Comments': 22, 'Shares': 3},
    'max': {'Views': 7_105_469, 'Likes': 821_805, 'Comments': 137_075, 'Shares': 75_864},
})

downloaded_engagement = pd.DataFrame({
    label: {
        'n': gesis[col].count(),
        'mean': gesis[col].mean(),
        'std': gesis[col].std(),
        'min': gesis[col].min(),
        'median': gesis[col].median(),
        'max': gesis[col].max(),
    }
    for label, col in engagement_columns.items()
}).T
display(downloaded_engagement.round(2))

agreement = downloaded_engagement[['mean', 'std', 'median', 'max']].round(2).eq(paper_engagement.round(2))
print('Every reported engagement value matches the paper after rounding to two decimals:', bool(agreement.to_numpy().all()))
display(agreement)

,n,mean,std,min,median,max
Views,"8,559.00","48,878.32","191,810.48",0.00,"1,645.00","7,105,469.00"
Likes,"8,623.00","4,523.64","17,199.79",1.00,707.00,"821,805.00"
Comments,"8,623.00",200.67,"1,655.49",0.00,22.00,"137,075.00"
Shares,"8,623.00",303.34,"1,854.64",0.00,3.00,"75,864.00"


Every reported engagement value matches the paper after rounding to two decimals: True


,mean,std,median,max
Views,True,True,True,True
Likes,True,True,True,True
Comments,True,True,True,True
Shares,True,True,True,True


### Distribution and concentration analysis

Social-media engagement is usually highly skewed. Means alone can therefore be misleading. This section compares mean and median and calculates how much of each metric comes from the top 1% of available videos.

In [13]:
def top_share(series, fraction=0.01):
    clean = series.dropna().sort_values(ascending=False)
    n_top = max(1, int(np.ceil(len(clean) * fraction)))
    return clean.iloc[:n_top].sum() / clean.sum() * 100

concentration = []
for label, col in engagement_columns.items():
    s = gesis[col].dropna()
    concentration.append({
        'metric': label,
        'mean': s.mean(),
        'median': s.median(),
        'mean_to_median_ratio': s.mean() / s.median() if s.median() else np.inf,
        'top_1_percent_share': top_share(s),
    })
display(pd.DataFrame(concentration).round(2))

,metric,mean,median,mean_to_median_ratio,top_1_percent_share
0,Views,"48,878.32","1,645.00",29.71,28.58
1,Likes,"4,523.64",707.00,6.40,25.20
2,Comments,200.67,22.00,9.12,34.08
3,Shares,303.34,3.00,101.11,44.23


### Countries, outlets, and hashtags

In [14]:
country_table = gesis['V4_country_of_origin'].value_counts().rename('videos').to_frame()
country_table['percent'] = country_table['videos'] / len(gesis) * 100
display(country_table.round(2))

outlet_table = gesis['V3_news_outlet'].value_counts().rename('videos').to_frame()
outlet_table['percent'] = outlet_table['videos'] / len(gesis) * 100
display(outlet_table)
print('Share of videos from the two largest outlets:', round(outlet_table.head(2)['percent'].sum(), 2), '%')
print('Share of videos from the five largest outlets:', round(outlet_table.head(5)['percent'].sum(), 2), '%')

,videos,percent
V4_country_of_origin,,
CH,4166,48.31
AT,3143,36.45
DE,1314,15.24


,videos,percent
V3_news_outlet,,
20minuten,1853,21.49
heute.at,1090,12.64
blick,754,8.74
watson_news,537,6.23
derstandard,444,5.15
zeitimbild,424,4.92
bluenews.ch,395,4.58
kleinezeitung,389,4.51
tagesschau,388,4.50


Share of videos from the two largest outlets: 34.13 %
Share of videos from the five largest outlets: 54.25 %


In [15]:
def parse_hashtags(value):
    if pd.isna(value):
        return [], 'missing'
    text = str(value).strip()
    if text.startswith('['):
        try:
            parsed = ast.literal_eval(text)
            return [str(x).strip().lower() for x in parsed if str(x).strip()], 'python-list'
        except (ValueError, SyntaxError):
            return [], 'malformed-list'
    return [x.strip().lower().lstrip('#') for x in text.split(',') if x.strip()], 'comma-delimited'

tag_counter = Counter()
format_counter = Counter()
tags_per_row = []
for value in gesis['M8_hashtags']:
    tags, storage_format = parse_hashtags(value)
    tag_counter.update(tags)
    format_counter[storage_format] += 1
    tags_per_row.append(len(tags))

hashtag_summary = pd.DataFrame([{
    'hashtag_occurrences': sum(tag_counter.values()),
    'unique_hashtags': len(tag_counter),
    'mean_hashtags_per_video': np.mean(tags_per_row),
    'median_hashtags_per_video': np.median(tags_per_row),
}])
display(hashtag_summary.round(3))
display(pd.DataFrame(tag_counter.most_common(20), columns=['hashtag', 'count']))
display(pd.DataFrame(format_counter.items(), columns=['stored_format', 'rows']).sort_values('rows', ascending=False))

,hashtag_occurrences,unique_hashtags,mean_hashtags_per_video,median_hashtags_per_video
0,52752,11674,6.12,5.00


,hashtag,count
0,news,2035
1,fyp,1723
2,nachrichten,1527
3,20min,1208
4,heutezeitung,943
5,tageszeitungheute,903
6,lernenmittiktok,898
7,fy,867
8,schweiz,815
9,österreich,588


,stored_format,rows
0,python-list,8559
1,comma-delimited,59
2,missing,5


## 4. Direct comparison

In [16]:
comparison = pd.DataFrame([
    {
        'dataset': 'US 2024 Presidential Election',
        'topic': 'U.S. election discussion',
        'downloaded_rows': int(us_profiles.loc[us_profiles.file == US_LATEST.name, 'rows'].iloc[0]),
        'accounts_or_creators': '693,826 in paper; not in public CSV',
        'public_structure': 'one exact video-ID column in latest release',
        'raw_video': 'No',
        'metadata': 'No public metadata in downloaded CSV',
        'annotations': 'No',
        'reconstruction': 'Hydration/API needed; IDs can decay; usernames absent',
        'collection': 'TikTok Research API; keyword/hashtag and date-phase queries',
    },
    {
        'dataset': 'News on TikTok (GESIS)',
        'topic': 'German-language professional news',
        'downloaded_rows': len(gesis),
        'accounts_or_creators': f"{gesis['V3_news_outlet'].nunique()} outlet accounts",
        'public_structure': '37 columns: IDs, URLs, metadata, engagement, annotations',
        'raw_video': 'No',
        'metadata': 'Yes',
        'annotations': '25 human-coded variables',
        'reconstruction': 'Exact ID and URL supplied; unavailable/private posts remain inaccessible',
        'collection': 'TikTok Research API by selected accounts; Zeeschuimer cross-check',
    },
])
display(comparison.T)

,0,1
dataset,US 2024 Presidential Election,News on TikTok (GESIS)
topic,U.S. election discussion,German-language professional news
downloaded_rows,4069908,8623
accounts_or_creators,"693,826 in paper; not in public CSV",18 outlet accounts
public_structure,one exact video-ID column in latest release,"37 columns: IDs, URLs, metadata, engagement, a..."
raw_video,No,No
metadata,No public metadata in downloaded CSV,Yes
annotations,No,25 human-coded variables
reconstruction,Hydration/API needed; IDs can decay; usernames...,Exact ID and URL supplied; unavailable/private...
collection,TikTok Research API; keyword/hashtag and date-...,TikTok Research API by selected accounts; Zees...


## 5. Main findings

1. The two releases have different research purposes. The U.S. dataset prioritizes scale, while GESIS prioritizes detailed metadata and manual content coding.
2. Neither downloaded dataset contains raw TikTok video or audio files.
3. The latest U.S. release contains exact video IDs, but no creator usernames or per-video metadata. Reconstruction requires another TikTok/API step and cannot recover deleted/private posts.
4. The U.S. paper and repository counts are not identical because the repository releases use later coverage windows. The files also overlap and must not be added together.
5. The U.S. `v2` file stores almost all IDs in scientific notation and has lost exact identifier precision. Use the February 2025 file for exact IDs.
6. The downloaded GESIS counts match the paper: 8,623 videos, 18 outlets, and 7,978 news videos. Its reported engagement statistics are also reproducible.
7. GESIS engagement is strongly right-skewed. A small group of popular videos accounts for a large share of views and interactions, so medians and concentration measures should be reported alongside means.
8. GESIS contains 64 missing view counts and mixed hashtag serialization formats. Analysis code should handle both issues explicitly.